In [13]:
import torch 
import torch.nn as nn
import torch.optim as optim
import numpy as np

text = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them. To die, to sleep;
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to: 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep, perchance to dream. Ay, there's the rub,
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil,
Must give us pause. There's the respect
That makes calamity of so long life.
""".strip()

text = text.lower()
chars = sorted(set(text))
char_to_idx = {char: i for i, char in enumerate(chars)}
idx_to_char = {i: char for i, char in enumerate(chars)}
vocab_size = len(chars)

print(f"文本长度: {len(text)}")
print(f"词汇表长度: {vocab_size}")
print(f"字符列表: {''.join(chars)}")

文本长度: 605
词汇表长度: 31
字符列表: 
 ',-.:;abcdefghiklmnopqrstuvwy


In [14]:
data = torch.tensor([char_to_idx[ch] for ch in text], dtype = torch.long)
print(f"data.shape: {data.shape}")
print(f"First 20 characters: {text[:20]}")
print(f"Corresponding numbers: {data[:20].tolist()}")

data.shape: torch.Size([605])
First 20 characters: to be, or not to be,
Corresponding numbers: [26, 21, 1, 9, 12, 3, 1, 21, 24, 1, 20, 21, 26, 1, 26, 21, 1, 9, 12, 3]


In [15]:
seq_length = 50

def create_sequences(data, seq_length):
    inputs = []
    targets = []
    for i in range(len(data) - seq_length):
        inputs.append(data[i : i+seq_length])
        targets.append(data[i + seq_length])
    return torch.stack(inputs), torch.stack(targets)

X, y = create_sequences(data, seq_length)
print(f"X type: {type(X)}")
print(f"X shape: {X.shape}")
print(f"X data type: {X.dtype}")
print(f"y type: {type(y)}")
print(f"y shape: {y.shape}")
print(f"y data type: {y.dtype}")


X type: <class 'torch.Tensor'>
X shape: torch.Size([555, 50])
X data type: torch.int64
y type: <class 'torch.Tensor'>
y shape: torch.Size([555])
y data type: torch.int64


In [16]:
print(len(text))

605


In [18]:
example_input = ''.join([idx_to_char[i.item()] for i in X[0]])
example_target = idx_to_char[y[0].item()]
print(f"\nInput: '{example_input}'")
print(f"Output: '{example_target}'")



Input: 'to be, or not to be, that is the question:
whether'
Output: ' '


In [27]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers = num_layers, batch_first = True, dropout = 0.2 if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden = None):
        # X shape: (batch, seq_len)
        embed = self.embedding(x) # Shape: (batch, seq_len, embed_size)
        # output shape: (batch, seq_len, hidden_size), hidden shape, 2 elements tuple, each element shape (num_layers, batch, hidden_size)
        output, hidden = self.lstm(embed, hidden) 
        output = output[:, -1, :] # output shape change to (batch, hidden_size)
        logits = self.fc(output) # logits shape (batch, vocab_size)
        return logits, hidden

    def init_hidden(self, batch_size):
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        return (h0, c0)

embed_size = 32
hidden_size = 128
model = CharLSTM(vocab_size, embed_size, hidden_size)
print(model)
print(f"number of parameters: {sum(p.numel() for p in model.parameters()):,}")

CharLSTM(
  (embedding): Embedding(31, 32)
  (lstm): LSTM(32, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=31, bias=True)
)
number of parameters: 87,935


In [30]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size = 64, shuffle=True)
optimizer = optim.Adam(model.parameters(), lr = 0.003)
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 50
print(f"{'Epoch':>5} | {'Loss':>8} | {'Sample':>50}")
print("-"*50)


def generate_text(model, char_to_idx, idx_to_char, seed = "to ", length = 100, temperature = 0.1):
    model.eval()
    input_seq = torch.tensor([char_to_idx[ch] for ch in seed]).unsqueeze(0)
    generated = seed
    hidden = None

    with torch.no_grad():
        for i in range(len(seed) - 1):
            single_char = input_seq[:, i:i+1]
            _, hidden = model(single_char, hidden)

            current_input = input_seq[:, -1:]
            
        for _ in range(length):
            logits, hidden = model(current_input, hidden)
            probs = torch.softmax(logits / temperature, dim=-1)

            next_idx = torch.multinomial(probs, 1)
            next_char = idx_to_char[next_idx.item()]

            generated += next_char
            current_input = next_idx
    return generated   

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0
    count = 0

    for batch_x, batch_y in loader:
        logits, _ = model(batch_x)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 5.0)
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        count += batch_x.size(0)

    avg_loss = total_loss / count

    if epoch == 1 or epoch % 10 == 0:
        sample = generate_text(model, char_to_idx, idx_to_char, seed="to be", length = 40)
        print(f"{epoch:>5} | {avg_loss:>8.4f} | {sample}")


         
            
        

        





Epoch |     Loss |                                             Sample
--------------------------------------------------
    1 |   0.0921 | to be wish'd th in shepis to die, to sleep;
n
   10 |   0.0108 | to be wish'd the thousand natural shocks
that
   20 |   0.0435 | to be wish'd. to die, to sleep;
to sleep, per
   30 |   0.0039 | to be wish'd. to die, to sleep;
to sleep, per
   40 |   0.0024 | to be wish'd. to die, to sleep;
to sleep, per
   50 |   0.0017 | to be wish'd. to die, to sleep;
to sleep, per


In [32]:
print("=== Temperature对比 ===\n")
for temp in [0.3, 0.8, 1.2]:
    text_out = generate_text(model, char_to_idx, idx_to_char,
                             seed="to be", length=100, temperature=temp)
    print(f"Temperature={temp}:")
    print(f"  {text_out}")
    print()

=== Temperature对比 ===

Temperature=0.3:
  to be wish'd. to die, to sleep;
to sleep, perchance to dream. ay, there's the rub,
for in that sleep of d

Temperature=0.8:
  to be of there's the respect
that makes calamity of so long life. to die, to sleep;
no more; and by a sle

Temperature=1.2:
  to be wheis to dream. ay, there's the rub,
for in that sleep of death what dreams may come
when we have s



In [33]:
print(generate_text(model, char_to_idx, idx_to_char, seed="to be", length=200, temperature=0.8))

to be wiinst a consummation
devoutly to be wish'd. to die, to sleep;
to sleep, perchance to dream. ay, there's the rub,
for in that sleep of death what dreams may come
when we have shuffled off this mortal


In [35]:
print(generate_text(model, char_to_idx, idx_to_char, seed="to be", length=200, temperature=0.8))

to be wherthanc the mind to suffer
the slings and arrows of outrageous fortune,
or to take arms against a sea of troubles,
and by opposing end them. to die, to sleep;
no more; and by a sleep to say we end



In [37]:
print(generate_text(model, char_to_idx, idx_to_char, seed="to be", length=200, temperature=1.2))

to be coupaatiso of outrageous fortune,
or to take arms against a sea of troubles,
and by opposing end them. 'tor al she ind to die, to sleep;
no more; and by a sleep to say we end
the heart-ache and the t


In [39]:
seeds = ["to be", "the", "and", "whether"]
for seed in seeds:
    text_out = generate_text(model, char_to_idx, idx_to_char, seed = seed, length=80, temperature=0.8)
    print(f"seed = {seed}, text_out = {text_out}")
    print()

seed = to be, text_out = to be wind to suffer
the slings and arrows of outrageous fortune,
or to take arms aga

seed = the, text_out = the whe's the respect
that makes calamity of so long life. to die, to sleep;
to sle

seed = and, text_out = and by opposing end them. to die, to sleep;
no more; and by a sleep to say we end
t

seed = whether, text_out = whether to die, to sleep;
no more; and by a sleep to say we end
the heart-ache and the 



In [41]:
torch.save({
    'model_state_dict': model.state_dict(),
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size,
    'embed_size': embed_size,
    'hidden_size': hidden_size
}, 'char_lstm_checkpoint.pth')
print("Model saved!")

Model saved!


In [42]:
checkpoint = torch.load('char_lstm_checkpoint.pth')
char_to_idx = checkpoint['char_to_idx']
idx_to_char = checkpoint['idx_to_char']
vocab_size = checkpoint['vocab_size']
embed_size = checkpoint['embed_size']
hidden_size = checkpoint['hidden_size']

model_loaded = CharLSTM(vocab_size, embed_size, hidden_size)
model_loaded.load_state_dict(checkpoint['model_state_dict'])

text = generate_text(model_loaded, char_to_idx, idx_to_char, seed = "to be", length=100)
print(text)

to be wish'd. to die, to sleep;
to sleep, perchance to dream. ay, there's the rub,
for in that sleep of d


In [ ]:
text = generate_text(model_loaded, char_to_idx, idx_to_char, seed = "to be", length=100)
print(text)